# DermaCare AI — Pipeline Hướng 1 (Coarse-to-Fine)
### YOLO ROI Detection → SegFormer Segmentation → MASI Scoring

Notebook này chạy trọn vẹn trên Google Colab (miễn phí, có GPU).

**Trước khi bắt đầu:** vào menu `Runtime > Change runtime type > GPU` để bật GPU miễn phí.

**Thứ tự thực hiện trong notebook này:**
1. Kiểm tra GPU, cài thư viện
2. Kết nối Google Drive (lưu dữ liệu/checkpoint lâu dài, tránh mất khi Colab ngắt kết nối)
3. Giai đoạn A: chuẩn bị dữ liệu (MultiCaRe + ISIC)
4. Giai đoạn B: xây & huấn luyện pipeline (Stage 1 YOLO + Stage 2 SegFormer)
5. Giai đoạn C: tích hợp MASI, chạy demo end-to-end


## 1. Kiểm tra GPU và cài đặt thư viện

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics transformers datasets accelerate \
    opencv-python-headless multiversity albumentations
import torch
print("CUDA khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Kết nối Google Drive (khuyến nghị — tránh mất dữ liệu khi phiên Colab hết hạn)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/DermaCare_Pipeline"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/scripts", exist_ok=True)
%cd {PROJECT_DIR}
print("Thư mục làm việc:", os.getcwd())


## 3. Giai đoạn A — Chuẩn bị dữ liệu

### 3.1. Tải & lọc ảnh Melasma thật từ MultiCaRe Dataset

In [ ]:
%%writefile scripts/01_fetch_melasma_images.py
"""
Script: 01_fetch_melasma_images.py
Muc dich: Loc va tai cac anh lien quan den benh Nam (Melasma) tu bo du lieu
MultiCaRe (130,000+ anh y khoa trich xuat tu cac bai bao case report mo
tren PubMed Central).

LICH SU LOI (giu lai de tham khao khi viet phan han che trong khoa luan):
Thu vien 'multiversity' mac dinh tai toan bo dataset qua endpoint gop file
cua Zenodo (/files-archive). Endpoint nay HIEN DANG LOI/BI HAN CHE phia
Zenodo (tra ve HTTP 400) cho du dung version_id nao -- day la loi tu dich vu
Zenodo, khong phai do cau hinh. GIAI PHAP: tai TUNG FILE LE qua API chuan
cua Zenodo (van hoat dong binh thuong), dat dung vao thu muc ma thu vien
mong doi, de no bo qua buoc tu tai bi loi va dung thang du lieu co san.

LUU Y:
- Chi chay duoc tren may/Colab co ket noi Internet binh thuong.
- Can dung luong trong toi thieu ~5GB (file nen + sau khi giai nen).
- Dataset MultiCaRe co giay phep tong the CC BY-NC-SA (phi thuong mai);
  TUNG anh rieng le co the co giay phep khac (CC-BY, CC-BY-NC, CC0) --
  kiem tra metadata truoc khi dua anh vao khoa luan.

Cach chay:
    pip install multiversity
    python 01_fetch_melasma_images.py
"""

import os
import shutil
import requests
from tqdm import tqdm
from multiversity.multicare_dataset import MedicalDatasetCreator


def warn_if_not_on_drive():
    """Canh bao neu dang chay trong o dia tam cua Colab (/content/, ngoai
    Google Drive) -- du lieu tai ve se BI XOA neu runtime bi ngat/khoi dong
    lai. Day la nguyen nhan pho bien nhat gay mat du lieu da tai."""
    cwd = os.getcwd()
    is_colab = os.path.exists("/content")
    if is_colab and "/drive/MyDrive" not in cwd:
        print("=" * 70)
        print("CANH BAO QUAN TRONG: ban dang KHONG chay trong Google Drive!")
        print(f"    Thu muc hien tai: {cwd}")
        print("    Du lieu tai ve se bi XOA neu Colab runtime bi ngat/restart.")
        print("    -> Chay lai cell 'Ket noi Google Drive' (mount + %cd) o dau")
        print("       notebook TRUOC KHI tiep tuc, de dam bao dang o dung")
        print("       duong dan /content/drive/MyDrive/DermaCare_Pipeline/")
        print("=" * 70)
        response = input("Van tiep tuc chay du sao (khong khuyen nghi)? (y/N): ")
        if response.strip().lower() != "y":
            raise SystemExit("Da dung lai. Chay lai cell mount Google Drive truoc.")


DATA_DIR = "medical_datasets"
DATASET_NAME = "melasma_dataset"
WHOLE_DATASET_PATH = f"{DATA_DIR}/whole_multicare_dataset"

# Ban ghi Zenodo da xac minh con hoat dong (Version 2.1, 2.9GB). Neu ban ghi
# nay sau nay cung ngung hoat dong, tim ban ghi moi hon tai:
# https://github.com/mauro-nievoff/MultiCaRe_Dataset (muc "Useful Resources")
VERSION_ID = "14994046"


def download_files_individually(version_id: str, dest_dir: str):
    """Tai TUNG FILE trong ban ghi Zenodo (bo qua endpoint /files-archive bi loi),
    roi tu giai nen cac file .zip -- lam thay cho buoc _download_dataset() bi
    loi ben trong thu vien multiversity."""
    os.makedirs(dest_dir, exist_ok=True)

    print(f">>> Lay danh sach file tu Zenodo record {version_id}...")
    resp = requests.get(f"https://zenodo.org/api/records/{version_id}")
    resp.raise_for_status()
    record = resp.json()
    files = record.get("files", [])
    if not files:
        raise RuntimeError(
            f"Khong lay duoc danh sach file tu record {version_id}. "
            f"Kiem tra lai ID hoac thu ban ghi khac."
        )
    print(f"    Tim thay {len(files)} file: {[f['key'] for f in files]}")

    downloaded_zip_paths = []
    for f in files:
        filename = f["key"]
        # Zenodo co 2 kieu cau truc link tuy thoi diem ban ghi duoc tao:
        # kieu cu dung 'links.self' de tai truc tiep, kieu moi can them
        # 'links.content'. Uu tien 'content' neu co, fallback ve 'self'.
        links = f.get("links", {})
        url = links.get("content") or links.get("self")
        size_mb = f.get("size", 0) / 1e6
        dest_path = os.path.join(dest_dir, filename)

        if os.path.exists(dest_path) and os.path.getsize(dest_path) > 1024:
            print(f"    Da co san {filename}, bo qua tai lai.")
        else:
            print(f"    Dang tai {filename} ({size_mb:.1f} MB)...")
            with requests.get(url, stream=True) as r:
                r.raise_for_status()
                # Kiem tra content-type truoc khi luu -- neu Zenodo tra ve
                # trang loi (html/json) thay vi file that, dung lai ngay
                # thay vi luu nham nhu loi da gap truoc do.
                content_type = r.headers.get("content-type", "")
                if "html" in content_type or "json" in content_type:
                    raise RuntimeError(
                        f"URL cho {filename} tra ve noi dung khong hop le "
                        f"(content-type={content_type}). Zenodo co the dang gap "
                        f"su co. Thu lai sau vai phut, hoac bao loi nay lai cho Claude."
                    )
                total = int(r.headers.get("content-length", 0))
                with open(dest_path, "wb") as out, tqdm(
                    total=total, unit="B", unit_scale=True, desc=filename
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        out.write(chunk)
                        pbar.update(len(chunk))

        if filename.lower().endswith(".zip"):
            downloaded_zip_paths.append(dest_path)

    print(">>> Giai nen cac file .zip...")
    for zip_path in downloaded_zip_paths:
        print(f"    Giai nen {zip_path}...")
        shutil.unpack_archive(zip_path, dest_dir)
        os.remove(zip_path)

    print(">>> Hoan tat tai va giai nen du lieu goc MultiCaRe.")


def main():
    warn_if_not_on_drive()

    # Buoc 1: neu thu muc du lieu goc chua co gi, tu tai thu cong (bo qua
    # endpoint bi loi cua thu vien). Neu da co roi (vd chay lai lan 2), bo qua.
    if not os.path.exists(WHOLE_DATASET_PATH) or not os.listdir(WHOLE_DATASET_PATH):
        download_files_individually(VERSION_ID, WHOLE_DATASET_PATH)
    else:
        print(f">>> Du lieu goc da co san tai {WHOLE_DATASET_PATH}, bo qua buoc tai.")

    # Buoc 2: khoi tao MedicalDatasetCreator -- vi thu muc da co san du lieu,
    # thu vien se TU DONG BO QUA buoc tu tai (_download_dataset) bi loi, va
    # di thang vao doc file de xu ly.
    print(">>> Buoc 2: Khoi tao MedicalDatasetCreator (doc du lieu da co san)...")
    mdc = MedicalDatasetCreator(directory=DATA_DIR, version_id=VERSION_ID)

    print(">>> Buoc 3: Dinh nghia bo loc tu khoa lien quan Melasma...")
    filters = [
        {
            "field": "case_strings",
            "string_list": ["melasma", "chloasma", "hyperpigmentation", "hypermelanosis"],
            "operator": "any",
        },
        {
            "field": "caption",
            "string_list": ["melasma", "chloasma", "hyperpigmentation", "facial pigmentation"],
            "operator": "any",
        },
    ]

    print(">>> Buoc 4: Tao subset 'melasma_dataset' (loai 'multimodal': anh + case text + caption)...")
    output_path = os.path.join(DATA_DIR, DATASET_NAME)
    if os.path.exists(output_path) and os.listdir(output_path):
        print(f"    Da co san dataset '{DATASET_NAME}' tai {output_path} (tu lan chay truoc).")
        print("    Bo qua buoc tao lai -- neu muon tao moi hoan toan, xoa thu muc nay roi chay lai.")
    else:
        mdc.create_dataset(
            dataset_name=DATASET_NAME,
            filter_list=filters,
            dataset_type="multimodal",
        )
    print(f"\nHOAN TAT. Du lieu da luu tai: {output_path}")
    print("   QUAN TRONG VE GIAY PHEP: dataset MultiCaRe co giay phep tong the")
    print("   CC BY-NC-SA (phi thuong mai), nhung TUNG anh rieng le co the co")
    print("   giay phep khac nhau (CC-BY, CC-BY-NC, CC-BY-NC-SA, hoac CC0).")
    print("   Kiem tra cot 'license' trong file metadata di kem TUNG anh truoc")
    print("   khi dua vao khoa luan, va trich dan dung nguon bai bao goc.")


if __name__ == "__main__":
    main()


In [ ]:
!python scripts/01_fetch_melasma_images.py

### 3.2. Tải bộ ISIC 2018 Task 1

**Bắt buộc thao tác thủ công của bạn:**
1. Truy cập https://challenge.isic-archive.com/data/#2018 (hoặc tìm "ISIC 2018 Task 1" trên Kaggle),
   tạo tài khoản miễn phí, tải các file Input + GroundTruth (Training và/hoặc Validation và/hoặc Test —
   script bước sau sẽ tự động gộp bất kỳ tổ hợp nào bạn có, không bắt buộc phải đủ cả 3 tập).
2. Upload các file zip này lên Google Drive, vào đúng thư mục `DermaCare_Pipeline/isic2018_raw/`
   (không cần giải nén tay — script bước sau sẽ TỰ ĐỘNG giải nén mọi file .zip tìm thấy).

In [ ]:
import os
os.makedirs("isic2018_raw", exist_ok=True)
print("Danh sách file hiện có trong isic2018_raw/:")
for f in os.listdir("isic2018_raw"):
    print(" -", f)
print("\n(Việc giải nén sẽ được thực hiện tự động ở script 02 bên dưới, không cần làm tay ở đây.)")


In [ ]:
%%writefile scripts/02_prepare_isic_dataset.py
"""
Script: 02_prepare_isic_dataset.py
Muc dich: Tien xu ly bo du lieu ISIC 2018 Task 1 (Lesion Boundary Segmentation)
de chuan bi huan luyen SegFormer o Giai doan 2.

YEU CAU TRUOC KHI CHAY:
1. Tai thu cong tu ISIC/Kaggle, dat cac file .zip vao isic2018_raw/ -- CHAP
   NHAN BAT KY TO HOP nao trong: Training/Validation/Test Input + GroundTruth
   (khong bat buoc phai co du ca 3 tap -- script se TU DONG GOM TAT CA cap
   Input+GroundTruth co san lai voi nhau, roi TU CHIA train/val theo ty le
   rieng, thay vi bat buoc theo dung quy uoc split cua ISIC challenge goc).
2. Script se TU DONG giai nen moi file .zip tim thay trong isic2018_raw/.

GHI CHU CHO KHOA LUAN: neu ban dung ca 3 tap con (Training/Validation/Test)
gop lai va tu chia train/val rieng, can neu ro trong Chuong 4.5 rang cach
chia nay KHAC voi quy uoc split chinh thuc cua ISIC 2018 Challenge (chi anh
huong ve mat "chuan hoa de so sanh voi leaderboard challenge", KHONG anh
huong tinh hop le cua so lieu huan luyen/danh gia rieng cua khoa luan).

Script se: resize anh + mask ve cung kich thuoc, chuan hoa mask ve {0,255},
chia train/val (85/15), luu vao thu muc processed_isic/ san sang cho buoc
huan luyen.
"""

import os
import shutil
import random
import zipfile
from datetime import datetime
from pathlib import Path
import numpy as np
from PIL import Image
from tqdm import tqdm

RAW_DIR = Path("isic2018_raw")

OUT_DIR = Path("processed_isic")
IMG_SIZE = 512
VAL_RATIO = 0.15
SEED = 42


def warn_if_not_on_drive():
    """Canh bao neu dang chay trong o dia tam cua Colab (/content/, ngoai
    Google Drive) -- du lieu se bi XOA neu runtime bi ngat/khoi dong lai."""
    cwd = os.getcwd()
    is_colab = os.path.exists("/content")
    if is_colab and "/drive/MyDrive" not in cwd:
        print("=" * 70)
        print("CANH BAO QUAN TRONG: ban dang KHONG chay trong Google Drive!")
        print(f"    Thu muc hien tai: {cwd}")
        print("    -> Chay lai cell 'Ket noi Google Drive' (mount + %cd) truoc.")
        print("=" * 70)
        response = input("Van tiep tuc chay du sao (khong khuyen nghi)? (y/N): ")
        if response.strip().lower() != "y":
            raise SystemExit("Da dung lai. Chay lai cell mount Google Drive truoc.")


def auto_extract_all_zips(root: Path):
    """Tu dong giai nen MOI file .zip tim thay truc tiep trong root (khong
    can biet truoc ten chinh xac cua tung file). LUON giai nen vao 1 thu muc
    con co dinh dat theo ten file zip (root/ten_zip/...) -- BAT KE cau truc
    ben trong zip co san san mot thu muc bao ngoai hay khong (co zip tu tao
    thu muc con trung ten, co zip lai giai nen thang cac file roi vao goc,
    tuy nha cung cap dong goi khac nhau). Cach nay dam bao vi tri sau cung
    luon nhat quan, du dinh dang zip dau vao co the khac nhau."""
    zip_files = list(root.glob("*.zip"))
    if not zip_files:
        return
    print(f">>> Tim thay {len(zip_files)} file .zip, dang giai nen...")
    for zip_path in zip_files:
        target_dir = root / zip_path.stem
        if target_dir.exists() and any(target_dir.iterdir()):
            print(f"    {zip_path.name}: da giai nen truoc do, bo qua.")
            continue
        print(f"    Giai nen {zip_path.name} -> {target_dir.name}/...")
        target_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(target_dir)


def find_dir(root: Path, keywords: list, required_ext: str) -> Path:
    """Tu dong do tim thu muc con co ten khop voi TAT CA tu khoa (khong phan
    biet hoa/thuong) VA thuc su chua file dinh dang required_ext ngay ben
    trong no -- tranh chon nham thu muc cha rong."""
    candidates = [p for p in root.rglob("*") if p.is_dir()]
    matched_with_files = []
    for cand in candidates:
        name_lower = cand.name.lower()
        if all(kw.lower() in name_lower for kw in keywords):
            file_count = len(list(cand.glob(f"*{required_ext}")))
            if file_count > 0:
                matched_with_files.append((cand, file_count))
    if not matched_with_files:
        return None
    matched_with_files.sort(key=lambda x: x[1], reverse=True)
    return matched_with_files[0][0]


def resolve_isic_pairs():
    """Tim TAT CA cap (thu muc anh, thu muc mask) co san cho tung split
    (Training/Validation/Test) -- CHI lay split nao co DU CA HAI (Input va
    GroundTruth), roi tra ve danh sach cac cap tim duoc de gop chung lai."""
    pairs = []
    split_counts = {}
    for split in ["training", "validation", "test"]:
        image_dir = find_dir(RAW_DIR, ["task1", split, "input"], ".jpg")
        mask_dir = find_dir(RAW_DIR, ["task1", split, "groundtruth"], ".png")
        if image_dir and mask_dir:
            n = len(list(image_dir.glob("*.jpg")))
            print(f">>> Tim thay du cap Input+GroundTruth cho tap '{split}': {n} anh")
            pairs.append((image_dir, mask_dir))
            split_counts[split] = n
        else:
            status = []
            if not image_dir:
                status.append("thieu Input")
            if not mask_dir:
                status.append("thieu GroundTruth")
            print(f"    Tap '{split}': {', '.join(status)} -- bo qua tap nay.")
            split_counts[split] = None

    if not pairs:
        print(f"\n>>> KHONG TIM DUOC cap Input+GroundTruth nao hop le. "
              f"Cau truc thuc te trong {RAW_DIR.resolve()}:")
        for p in sorted(RAW_DIR.rglob("*")):
            if p.is_dir():
                print(f"    [DIR]  {p}")
        raise FileNotFoundError(
            "Khong tim thay du lieu ISIC hop le. Xem danh sach thu muc o tren, "
            "gui lai cho Claude de duoc ho tro neu can."
        )
    return pairs, split_counts


def resize_pair(img_path: Path, mask_path: Path, size: int):
    img = Image.open(img_path).convert("RGB").resize((size, size), Image.BILINEAR)
    mask = Image.open(mask_path).convert("L").resize((size, size), Image.NEAREST)
    mask_arr = (np.array(mask) > 127).astype(np.uint8) * 255
    return img, Image.fromarray(mask_arr)


def main():
    warn_if_not_on_drive()

    # Xoa sach OUT_DIR truoc khi xu ly, tranh lan du lieu cu tu lan chay
    # truoc (vd neu isic2018_raw thay doi giua cac lan chay, du lieu cu se
    # khong tu bien mat neu khong xoa thu cong -- day chinh la nguyen nhan
    # gay sai lech so lieu da gap phai truoc do).
    if OUT_DIR.exists():
        print(f">>> Xoa du lieu da xu ly cu tai {OUT_DIR} truoc khi lam lai tu dau...")
        shutil.rmtree(OUT_DIR)

    auto_extract_all_zips(RAW_DIR)
    pairs, split_counts = resolve_isic_pairs()

    # Gop TOAN BO cap (anh, mask) tu moi split tim duoc lai thanh 1 danh sach
    # duy nhat, roi TU CHIA train/val theo ty le rieng (xem ghi chu dau file).
    all_image_mask_pairs = []
    for image_dir, mask_dir in pairs:
        for img_path in sorted(image_dir.glob("*.jpg")):
            mask_path = mask_dir / f"{img_path.stem}_segmentation.png"
            if mask_path.exists():
                all_image_mask_pairs.append((img_path, mask_path))

    print(f"\n>>> TONG CONG gom duoc {len(all_image_mask_pairs)} cap anh+mask hop le "
          f"tu {len(pairs)} tap con.")
    assert len(all_image_mask_pairs) > 0, "Khong gom duoc cap anh+mask nao hop le."

    random.seed(SEED)
    random.shuffle(all_image_mask_pairs)
    n_val = int(len(all_image_mask_pairs) * VAL_RATIO)
    val_pairs = set(p[0] for p in all_image_mask_pairs[:n_val])
    n_train = len(all_image_mask_pairs) - n_val

    for split in ["train", "val"]:
        (OUT_DIR / split / "images").mkdir(parents=True, exist_ok=True)
        (OUT_DIR / split / "masks").mkdir(parents=True, exist_ok=True)

    corrupted_files = []
    processed_count = 0
    for img_path, mask_path in tqdm(all_image_mask_pairs, desc="Xu ly anh"):
        stem = img_path.stem
        split = "val" if img_path in val_pairs else "train"
        try:
            img, mask = resize_pair(img_path, mask_path, IMG_SIZE)
        except Exception as e:
            corrupted_files.append((str(img_path), str(e)))
            continue
        img.save(OUT_DIR / split / "images" / f"{stem}.jpg", quality=95)
        mask.save(OUT_DIR / split / "masks" / f"{stem}.png")
        processed_count += 1

    if corrupted_files:
        print(f"\n[!] Bo qua {len(corrupted_files)} anh loi/hong (khong doc duoc):")
        for path, err in corrupted_files[:10]:
            print(f"    - {path} ({err})")
        if len(corrupted_files) > 10:
            print(f"    ... va {len(corrupted_files) - 10} anh loi khac")
        # Cap nhat lai n_train/n_val cho dung so anh THAT SU da xu ly thanh cong
        n_val = sum(1 for p, _ in all_image_mask_pairs
                    if p in val_pairs and p not in [Path(c[0]) for c in corrupted_files])
        n_train = processed_count - n_val

    print(f"\nHOAN TAT. Train: {n_train} anh | Val: {n_val} anh")
    print(f"   Du lieu da xu ly luu tai: {OUT_DIR.resolve()}")

    # Ghi "bien ban du lieu" -- luu lai chinh xac da dung nhung tap con nao,
    # bao nhieu anh, luc nao -- de tranh tai dien tinh huong khong biet ro
    # chinh xac da huan luyen tren bao nhieu anh (da tung gap phai truoc do).
    manifest_path = Path("DATASET_MANIFEST.txt")
    manifest_lines = [
        f"Bien ban du lieu ISIC -- tao luc: {datetime.now().isoformat(timespec='seconds')}",
        f"Thu muc nguon: {RAW_DIR.resolve()}",
        "",
        "Cac tap con da tim thay va su dung:",
    ]
    for split, n in split_counts.items():
        status = f"{n} anh" if n is not None else "KHONG CO (bi bo qua)"
        manifest_lines.append(f"  - {split}: {status}")
    manifest_lines += [
        "",
        f"TONG SO CAP ANH+MASK TIM DUOC (truoc khi loc anh loi): {len(all_image_mask_pairs)}",
        f"So anh loi/hong bi bo qua khi xu ly: {len(corrupted_files)}",
        f"TONG SO ANH THUC TE DA XU LY THANH CONG: {processed_count}",
        f"Chia train/val (ty le {1-VAL_RATIO:.0%}/{VAL_RATIO:.0%}, random seed={SEED},",
        f"tinh tren so anh xu ly thanh cong, khong tinh anh loi):",
        f"  - Train: {n_train} anh",
        f"  - Val:   {n_val} anh",
        "",
        "LUU Y: file nay duoc tao lai MOI LAN chay script 02 -- neu chay lai",
        "voi du lieu trong isic2018_raw/ khac di, so lieu o day se thay doi theo.",
        "Luon doi chieu file nay voi so lieu ghi trong bao cao truoc khi nop.",
    ]
    if corrupted_files:
        manifest_lines.append("\nDanh sach anh loi/hong (khong tinh vao tong so):")
        for path, err in corrupted_files:
            manifest_lines.append(f"  - {path} ({err})")
    manifest_path.write_text("\n".join(manifest_lines), encoding="utf-8")
    print(f"   Da ghi bien ban du lieu tai: {manifest_path.resolve()}")


if __name__ == "__main__":
    main()


In [ ]:
!python scripts/02_prepare_isic_dataset.py

## 4. Giai đoạn B — Giai đoạn 1: YOLO Face Detection + Suy luận ROI

**Tải model YOLO-face pretrained** (đã xác minh nguồn hoạt động — `Bingsu/adetailer`
trên HuggingFace, huấn luyện trên bộ chuẩn WIDER FACE, giấy phép Apache-2.0):

In [ ]:
import os
os.makedirs("weights", exist_ok=True)

# Nguồn chính (đã xác minh hoạt động, 6.23MB, huấn luyện trên WIDER FACE):
!wget -q -O weights/yolov8n-face.pt \
    "https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8n.pt"

import os
size = os.path.getsize("weights/yolov8n-face.pt") if os.path.exists("weights/yolov8n-face.pt") else 0
if size > 1_000_000:
    print(f"Tải thành công ({size/1e6:.1f} MB)")
else:
    print("Tải thất bại hoặc file lỗi -- thử cell nguồn dự phòng bên dưới")


In [ ]:
# Nguồn dự phòng 1 (nếu nguồn chính lỗi):
!wget -q -O weights/yolov8n-face.pt \
    "https://github.com/lindevs/yolov8-face/releases/latest/download/yolov8n-face.pt"
import os
size = os.path.getsize("weights/yolov8n-face.pt") if os.path.exists("weights/yolov8n-face.pt") else 0
print(f"Kích thước file: {size/1e6:.1f} MB" if size > 1_000_000 else "Vẫn lỗi -- báo lại cho Claude kèm log")


In [ ]:
%%writefile scripts/03_stage1_roi_detection.py
"""
Script: 03_stage1_roi_detection.py
Muc dich: GIAI DOAN 1 cua pipeline Huong 1 -- dinh vi khung khuon mat bang
YOLO (pretrained tren tap WIDERFace), sau do suy luan hinh hoc ra 5 vung
giai phau hay bi Nam: Tran, Ma trai, Ma phai, Mui, Cam.

CACH LAY MODEL YOLO-FACE (da xac minh nguon hoat dong, tai file .pt, dat vao
thu muc weights/):
    https://huggingface.co/Bingsu/adetailer/resolve/main/face_yolov8n.pt
    (repo Bingsu/adetailer, huan luyen tren bo chuan WIDER FACE, giay phep
    Apache-2.0, dang duoc dung rong rai boi cong cu ADetailer)
    -> luu thanh weights/yolov8n-face.pt

Neu CHUA co file weights nay, script se TU DONG dung phuong an du phong
(OpenCV Haar Cascade -- da co san trong opencv-python, khong can tai them gi)
de pipeline luon chay duoc ngay, khong bi gian doan.

QUAN TRONG: Ty le hinh hoc (FOREHEAD_RATIO, CHEEK_RATIO...) ben duoi la uoc
luong dua tren ty le khuon mat co dien (rule of thirds/fifths) thuong dung
trong tai lieu da lieu/phau thuat tham my. Ban NEN kiem tra/hieu chinh lai
cac ty le nay bang tap anh da tu gan nhan tay (xem script 04_evaluate_stage1.py).
"""

from dataclasses import dataclass
from pathlib import Path
import cv2
import numpy as np

WEIGHTS_PATH = Path("weights/yolov8n-face.pt")

# Ty le hinh hoc uoc luong tung vung so voi khung mat (x1,y1,x2,y2)
FOREHEAD_TOP = 0.05      # bat dau tu 5% duoi dinh dau (tranh vung toc)
FOREHEAD_BOTTOM = 0.30
MIDFACE_BOTTOM = 0.68     # het vung mui/ma
CHIN_BOTTOM = 0.98
CHEEK_INNER = 0.38        # bien trong cua ma (gan mui)
CHEEK_OUTER = 0.90        # bien ngoai cua ma
NOSE_LEFT = 0.38
NOSE_RIGHT = 0.62
CHIN_LEFT = 0.30
CHIN_RIGHT = 0.70


@dataclass
class ROI:
    name: str
    box: tuple  # (x1, y1, x2, y2) toa do tuyet doi tren anh goc


class FaceDetector:
    """Wrapper: uu tien YOLO-face pretrained, tu dong fallback sang Haar Cascade."""

    def __init__(self):
        self.mode = None
        if WEIGHTS_PATH.exists():
            from ultralytics import YOLO
            self.model = YOLO(str(WEIGHTS_PATH))
            self.mode = "yolo"
            print(f"[FaceDetector] Dang dung YOLO-face: {WEIGHTS_PATH}")
        else:
            cascade_path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
            self.model = cv2.CascadeClassifier(cascade_path)
            self.mode = "haar_fallback"
            print(f"[FaceDetector] CANH BAO: khong tim thay {WEIGHTS_PATH}.")
            print("   Dang dung phuong an du phong Haar Cascade (do chinh xac thap hon YOLO).")
            print("   -> Tai yolov8n-face.pt tu link trong docstring de dung YOLO that.")

    def detect(self, image_bgr: np.ndarray):
        """Tra ve bounding box (x1,y1,x2,y2) cua khuon mat lon nhat trong anh,
        hoac None neu khong tim thay."""
        if self.mode == "yolo":
            results = self.model.predict(image_bgr, verbose=False)
            boxes = results[0].boxes
            if len(boxes) == 0:
                return None
            # chon box co confidence cao nhat
            best_idx = boxes.conf.argmax().item()
            x1, y1, x2, y2 = boxes.xyxy[best_idx].tolist()
            return int(x1), int(y1), int(x2), int(y2)
        else:
            gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
            faces = self.model.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
            if len(faces) == 0:
                return None
            # chon khuon mat co dien tich lon nhat
            x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
            return int(x), int(y), int(x + w), int(y + h)


def derive_anatomical_rois(face_box: tuple) -> list:
    """Suy luan 5 vung ROI (Tran, Ma trai, Ma phai, Mui, Cam) tu khung khuon mat,
    dua tren ty le hinh hoc dinh nghia o dau file."""
    x1, y1, x2, y2 = face_box
    w = x2 - x1
    h = y2 - y1

    def pt(px, py):
        return int(x1 + px * w), int(y1 + py * h)

    rois = [
        ROI("Tran", (*pt(0.10, FOREHEAD_TOP), *pt(0.90, FOREHEAD_BOTTOM))),
        ROI("Ma_trai", (*pt(1 - CHEEK_OUTER, FOREHEAD_BOTTOM), *pt(1 - CHEEK_INNER, MIDFACE_BOTTOM))),
        ROI("Ma_phai", (*pt(CHEEK_INNER, FOREHEAD_BOTTOM), *pt(CHEEK_OUTER, MIDFACE_BOTTOM))),
        ROI("Mui", (*pt(NOSE_LEFT, FOREHEAD_BOTTOM), *pt(NOSE_RIGHT, MIDFACE_BOTTOM))),
        ROI("Cam", (*pt(CHIN_LEFT, MIDFACE_BOTTOM), *pt(CHIN_RIGHT, CHIN_BOTTOM))),
    ]
    return rois


def crop_roi(image_bgr: np.ndarray, roi: ROI) -> np.ndarray:
    """Cat vung ROI, GIU NGUYEN do phan giai goc (khong resize) -- dung tinh
    than giai quyet 'resolution tradeoff' cua Huong 1."""
    x1, y1, x2, y2 = roi.box
    h_img, w_img = image_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w_img, x2), min(h_img, y2)
    return image_bgr[y1:y2, x1:x2]


def process_image(image_path: str, detector: FaceDetector, out_dir: str = "rois_output"):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Khong doc duoc anh: {image_path}")

    face_box = detector.detect(image)
    if face_box is None:
        print(f"[!] Khong phat hien khuon mat trong: {image_path}")
        return None

    rois = derive_anatomical_rois(face_box)

    out_path = Path(out_dir) / Path(image_path).stem
    out_path.mkdir(parents=True, exist_ok=True)

    for roi in rois:
        crop = crop_roi(image, roi)
        if crop.size == 0:
            continue
        cv2.imwrite(str(out_path / f"{roi.name}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 100])

    # Anh minh hoa: ve khung ROI len anh goc (dung cho bao cao khoa luan)
    vis = image.copy()
    for roi in rois:
        x1, y1, x2, y2 = roi.box
        cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(vis, roi.name, (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    cv2.imwrite(str(out_path / "visualization.jpg"), vis)

    print(f"[OK] {image_path} -> {out_path} ({len(rois)} vung ROI)")
    return rois


if __name__ == "__main__":
    import sys
    import json

    if len(sys.argv) < 2:
        print("Cach dung: python 03_stage1_roi_detection.py <duong_dan_anh.jpg> [...]")
        sys.exit(1)

    detector = FaceDetector()

    pred_path = Path("predictions_rois.json")
    predictions = {}
    if pred_path.exists():
        try:
            predictions = json.loads(pred_path.read_text(encoding="utf-8"))
        except Exception:
            pass

    for img_path in sys.argv[1:]:
        try:
            rois = process_image(img_path, detector)
        except FileNotFoundError as e:
            print(f"[!] Bo qua (khong doc duoc, co the da bi xoa/di chuyen): {img_path}")
            continue
        if rois is not None:
            predictions[img_path] = {roi.name: list(roi.box) for roi in rois}

    pred_path.write_text(json.dumps(predictions, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"\n[HOAN TAT] Da luu toa do du doan vao file: {pred_path} ({len(predictions)} anh)")


**Thử nghiệm nhanh Giai đoạn 1** (thay đường dẫn ảnh mẫu của bạn):

In [ ]:
# Upload thử 1 ảnh mặt mẫu vào Colab (dùng nút Files bên trái) rồi chạy:
# !python scripts/03_stage1_roi_detection.py duong_dan_anh_mau.jpg
print("Upload ảnh mẫu qua tab Files bên trái, rồi bỏ comment dòng lệnh phía trên và sửa đường dẫn.")


### Đánh giá IoU thật cho Giai đoạn 1

⚠️ **Lưu ý quan trọng:** bước "tự gán nhãn tay" (`annotate`) của script dưới đây
**KHÔNG chạy được trên Colab** (cần cửa sổ đồ hoạ, chuột). Bạn cần:
1. Tải file `scripts/04_evaluate_stage1.py` này về máy cá nhân
2. Chạy `python 04_evaluate_stage1.py annotate anh1.jpg anh2.jpg ...` TRÊN MÁY CÁ NHÂN
3. Upload file `ground_truth_rois.json` kết quả lên lại Colab/Drive
4. Sau đó mới chạy chế độ `evaluate` (tính toán thuần, chạy tốt trên Colab) ở cell bên dưới

In [ ]:
%%writefile scripts/04_evaluate_stage1.py
"""
Script: 04_evaluate_stage1.py
Muc dich: (a) Cong cu don gian de TU GAN NHAN TAY khung ROI chuan tren mot
tap nho anh mat (~20-30 anh), va (b) tinh IoU THAT giua ROI do model du
doan (script 03) so voi nhan tay -- day la SO LIEU DINH LUONG THAT cho
Giai doan 1 trong khoa luan (khong can cong cu ngoai nhu CVAT/Roboflow).

QUAN TRONG -- CHE DO 'annotate' CHI CHAY DUOC TREN MAY CA NHAN (KHONG chay
duoc tren Google Colab), vi no can mo cua so do hoa (cv2.imshow) va bat thao
tac chuot -- Colab la moi truong server, khong co man hinh hien thi. Neu ban
dang dung Colab: hay tai script nay + vai anh mau ve MAY CA NHAN, chay che do
'annotate' o do, roi upload file ground_truth_rois.json ket qua len lai
Colab/Google Drive de dung cho che do 'evaluate' (che do nay tinh toan thuan
tuy, chay binh thuong tren Colab).

CACH DUNG:

  Buoc 1 -- Tu gan nhan tay (CHAY TREN MAY CA NHAN, ve 5 khung bang chuot):
      python 04_evaluate_stage1.py annotate anh_mau/*.jpg

      Huong dan thao tac khi cua so hien len:
        - Keo chuot de ve khung cho tung vung, theo dung thu tu duoc in ra
          terminal (Tran -> Ma_trai -> Ma_phai -> Mui -> Cam)
        - Nhan 'n' (hoac Enter) de xac nhan khung vua ve, qua vung/anh tiep theo
        - Nhan 'x' de bo qua RIENG VUNG hien tai (dung khi vung nay khong hien
          dien/khong quan sat duoc trong anh, vd goc chup nghieng che khuat --
          CAC VUNG KHAC da ve dung trong CUNG anh nay VAN duoc giu lai)
        - Nhan 's' de BO QUA CA ANH HIEN TAI, bao gom CA CAC VUNG DA VE TRUOC DO
          trong anh nay cung bi huy (dung khi CA ANH khong phai anh mat -- vd
          anh soi da phong dai, mo hoc, bieu do...)
        - Nhan 'q' de dung han chuong trinh (nhung gi da luu truoc do van giu)
        - Chay lai lenh nay nhieu lan voi tap anh khac nhau se TU DONG BO QUA
          anh da gan nhan roi, chi hoi cac anh moi -- khong lo lam lai tu dau

  Buoc 2 -- Chay du doan cua model (script 03) tren cung tap anh do, luu
            ket qua vao predictions.json (xem huong dan cuoi file).

  Buoc 3 -- Tinh IoU that:
      python 04_evaluate_stage1.py evaluate
"""

import json
import sys
from pathlib import Path
import cv2

GT_PATH = Path("ground_truth_rois.json")
PRED_PATH = Path("predictions_rois.json")
REGION_ORDER = ["Tran", "Ma_trai", "Ma_phai", "Mui", "Cam"]


class SkipImageSignal(Exception):
    """Duoc nem ra khi nguoi dung nhan 's' de bo qua NGUYEN CA ANH hien tai
    (vd anh khong phai anh khuon mat -- soi da, mo hoc, bieu do...). Moi vung
    da ve truoc do trong anh nay (neu co) se KHONG duoc luu."""
    pass


class SkipRegionSignal(Exception):
    """Duoc nem ra khi nguoi dung nhan 'x' de bo qua RIENG VUNG hien tai (vd
    goc chup nghieng khien vung nay khong hien dien/khong quan sat duoc),
    trong khi VAN GIU LAI cac vung khac da ve dung cho anh nay."""
    pass


# ---------------------------------------------------------------------------
# CHE DO 1: TU GAN NHAN TAY
# ---------------------------------------------------------------------------
def annotate(image_paths):
    ground_truth = {}
    if GT_PATH.exists():
        ground_truth = json.loads(GT_PATH.read_text(encoding="utf-8"))

    skipped_bad_images = []
    for img_path in image_paths:
        img_path = str(img_path)

        if img_path in ground_truth:
            print(f"[Bo qua] Da gan nhan truoc do: {img_path}")
            continue

        image = cv2.imread(img_path)
        if image is None:
            print(f"[!] Bo qua, khong doc duoc: {img_path}")
            continue

        boxes = {}
        try:
            for region_name in REGION_ORDER:
                print(f"\n>>> Anh: {img_path} | Ve khung cho vung: {region_name}")
                print("    (Keo chuot trai de ve khung, tha chuot de xac nhan)")
                print("    Phim tat: 'n'=xac nhan | 'x'=vung nay KHONG hien dien"
                      " (giu lai cac vung khac) | 's'=BO QUA CA ANH | 'q'=dung han")
                try:
                    box = _draw_single_box(image.copy(), region_name)
                except SkipRegionSignal:
                    print(f"    -> Da bo qua rieng vung '{region_name}' (khong hien dien trong anh).")
                    continue
                if box is not None:
                    boxes[region_name] = list(box)
        except SkipImageSignal:
            print(f"[Da bo qua] Anh khong hop le (khong phai anh mat?): {img_path}")
            skipped_bad_images.append(img_path)
            cv2.destroyAllWindows()
            continue

        if not boxes:
            print(f"[!] Khong co vung nao hop le cho {img_path}, khong luu anh nay.")
            continue

        ground_truth[img_path] = boxes
        GT_PATH.write_text(json.dumps(ground_truth, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"[OK] Da luu nhan cho {img_path} ({len(boxes)}/5 vung) -> {GT_PATH}")

    cv2.destroyAllWindows()
    print(f"\nHOAN TAT. Da gan nhan tong cong {len(ground_truth)} anh (tich luy) trong {GT_PATH}.")
    if skipped_bad_images:
        print(f"Da bo qua {len(skipped_bad_images)} anh khong hop le (nhan 's'):")
        for p in skipped_bad_images:
            print(f"    - {p}")


def _draw_single_box(image, window_title):
    state = {"start": None, "box": None, "drawing": False}

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            state["start"] = (x, y)
            state["drawing"] = True
        elif event == cv2.EVENT_MOUSEMOVE and state["drawing"]:
            disp = image.copy()
            cv2.rectangle(disp, state["start"], (x, y), (0, 255, 0), 2)
            cv2.imshow(window_title, disp)
        elif event == cv2.EVENT_LBUTTONUP:
            state["drawing"] = False
            x1, y1 = state["start"]
            state["box"] = (min(x1, x), min(y1, y), max(x1, x), max(y1, y))

    cv2.namedWindow(window_title)
    cv2.setMouseCallback(window_title, on_mouse)
    cv2.imshow(window_title, image)

    while True:
        key = cv2.waitKey(20) & 0xFF
        if key == ord("n") or key == 13:  # 'n' hoac Enter -> xac nhan xong khung nay
            break
        if key == ord("x"):  # 'x' -> bo qua RIENG vung nay, giu lai cac vung khac
            cv2.destroyWindow(window_title)
            raise SkipRegionSignal()
        if key == ord("s"):  # 's' -> bo qua nguyen anh nay, khong luu gi ca
            cv2.destroyWindow(window_title)
            raise SkipImageSignal()
        if key == ord("q"):
            cv2.destroyWindow(window_title)
            sys.exit("Da dung theo yeu cau nguoi dung (q).")

    cv2.destroyWindow(window_title)
    return state["box"]


# ---------------------------------------------------------------------------
# CHE DO 2: TINH IoU THAT
# ---------------------------------------------------------------------------
def compute_iou(box_a, box_b):
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b

    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)

    area_a = (xa2 - xa1) * (ya2 - ya1)
    area_b = (xb2 - xb1) * (yb2 - yb1)
    union = area_a + area_b - inter_area
    return inter_area / union if union > 0 else 0.0


def evaluate():
    assert GT_PATH.exists(), f"Chua co {GT_PATH} -- chay 'annotate' truoc."
    assert PRED_PATH.exists(), (
        f"Chua co {PRED_PATH}. Ban can tu chay script 03 tren cung tap anh va luu "
        f"ket qua ROI vao file JSON cung cau truc nhu {GT_PATH}, dat ten {PRED_PATH}."
    )

    ground_truth = json.loads(GT_PATH.read_text(encoding="utf-8"))
    predictions = json.loads(PRED_PATH.read_text(encoding="utf-8"))

    per_region_ious = {r: [] for r in REGION_ORDER}
    for img_path, gt_boxes in ground_truth.items():
        pred_boxes = predictions.get(img_path, {})
        for region in REGION_ORDER:
            if region in gt_boxes and region in pred_boxes:
                iou = compute_iou(gt_boxes[region], pred_boxes[region])
                per_region_ious[region].append(iou)

    print("\n=== KET QUA DANH GIA GIAI DOAN 1 (IoU that, tinh tren tap tu gan nhan) ===")
    all_ious = []
    for region, ious in per_region_ious.items():
        if ious:
            mean_iou = sum(ious) / len(ious)
            all_ious.extend(ious)
            print(f"  {region:10s}: mean IoU = {mean_iou:.3f}  (n={len(ious)} anh)")
        else:
            print(f"  {region:10s}: khong co du lieu")

    if all_ious:
        print(f"\n  TRUNG BINH TOAN BO (mIoU): {sum(all_ious) / len(all_ious):.3f}")


if __name__ == "__main__":
    if len(sys.argv) < 2 or sys.argv[1] not in ("annotate", "evaluate"):
        print("Cach dung:")
        print("  python 04_evaluate_stage1.py annotate anh1.jpg anh2.jpg ...")
        print("  python 04_evaluate_stage1.py evaluate")
        sys.exit(1)

    if sys.argv[1] == "annotate":
        annotate(sys.argv[2:])
    else:
        evaluate()


In [ ]:
# Sau khi đã có ground_truth_rois.json (upload từ máy cá nhân) và predictions_rois.json:
# !python scripts/04_evaluate_stage1.py evaluate
print("Chạy dòng trên sau khi đã chuẩn bị đủ 2 file JSON cần thiết.")


## 5. Giai đoạn B — Giai đoạn 2: Huấn luyện SegFormer trên ISIC

In [ ]:
%%writefile scripts/05_stage2_train_segformer.py
"""
Script: 05_stage2_train_segformer.py
Muc dich: GIAI DOAN 2 cua pipeline Huong 1 -- fine-tune SegFormer (mo hinh
segmentation dang Transformer) tren bo du lieu ISIC 2018 (da qua tien xu ly
o script 02) de hoc phan vung tổn thuong da dang pixel-level.

Day la mo hinh duoc HUAN LUYEN THAT, co checkpoint that, co so lieu danh gia
that (xem script 06) -- dung lam thanh phan "lo AI" chinh cho Chuong 4 khoa
luan. Luu y: huan luyen tren ISIC (anh soi da, ton thuong khu tru) nhu mot
proxy dataset do THIEU du lieu Nam da co gan nhan public (da neu ro trong
Chuong 4.5 la han che phuong phap).

CHAY TREN GOOGLE COLAB: nho chon Runtime > Change runtime type > GPU truoc
khi chay script nay, neu khong se rat cham.
"""

from pathlib import Path
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from tqdm import tqdm

DATA_DIR = Path("processed_isic")
OUTPUT_DIR = Path("segformer_melasma_checkpoint")
MODEL_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"  # backbone pretrained nhe, phu hop fine-tune

NUM_EPOCHS = 15
BATCH_SIZE = 2  # giam tu 4 xuong 2 -- phu hop GPU VRAM thap (vd GTX 1050 Ti 4GB)
LEARNING_RATE = 6e-5
NUM_LABELS = 2  # 0 = nen (background/da thuong), 1 = vung ton thuong/sac to
GRADIENT_ACCUMULATION_STEPS = 2  # bu lai cho batch size nho: batch "hieu qua" = 2*2 = 4


def warn_if_not_on_drive():
    """Canh bao neu dang chay trong o dia tam cua Colab (/content/, ngoai
    Google Drive) -- checkpoint huan luyen se bi XOA neu runtime bi ngat.
    Day la buoc TON NHIEU THOI GIAN NHAT trong toan bo pipeline, nen kiem
    tra ky truoc khi bat dau huan luyen."""
    cwd = os.getcwd()
    is_colab = os.path.exists("/content")
    if is_colab and "/drive/MyDrive" not in cwd:
        print("=" * 70)
        print("CANH BAO QUAN TRONG: ban dang KHONG chay trong Google Drive!")
        print(f"    Thu muc hien tai: {cwd}")
        print("    Checkpoint huan luyen (co the mat vai gio de tao ra) se")
        print("    bi XOA neu Colab runtime bi ngat/restart giua chung.")
        print("    -> Chay lai cell 'Ket noi Google Drive' (mount + %cd) truoc.")
        print("=" * 70)
        response = input("Van tiep tuc chay du sao (khong khuyen nghi)? (y/N): ")
        if response.strip().lower() != "y":
            raise SystemExit("Da dung lai. Chay lai cell mount Google Drive truoc.")


class LesionSegDataset(Dataset):
    def __init__(self, split_dir: Path, processor: SegformerImageProcessor):
        self.image_dir = split_dir / "images"
        self.mask_dir = split_dir / "masks"
        self.image_files = sorted(self.image_dir.glob("*.jpg"))
        self.processor = processor

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        mask_path = self.mask_dir / f"{img_path.stem}.png"

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")
        mask_arr = (np.array(mask) > 127).astype(np.int64)  # {0, 1}

        encoded = self.processor(image, mask_arr, return_tensors="pt")
        return {
            "pixel_values": encoded["pixel_values"].squeeze(0),
            "labels": encoded["labels"].squeeze(0),
        }


def main():
    warn_if_not_on_drive()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f">>> Thiet bi su dung: {device}")
    if device.type == "cpu":
        print("    CANH BAO: dang chay tren CPU, se rat cham. Kiem tra lai da cai dung ban")
        print("    PyTorch co ho tro CUDA chua (xem HUONG_DAN_CAI_DAT_GPU.md).")
    else:
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"    GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM)")
        if vram_gb < 6:
            print(f"    LUU Y: VRAM thap ({vram_gb:.1f}GB) -- neu gap loi 'CUDA out of")
            print(f"    memory', giam them BATCH_SIZE xuong 1 hoac giam IMG_SIZE trong")
            print(f"    script 02_prepare_isic_dataset.py (vd 512 -> 384) roi xu ly lai du lieu.")

    processor = SegformerImageProcessor(do_reduce_labels=False)
    train_ds = LesionSegDataset(DATA_DIR / "train", processor)
    val_ds = LesionSegDataset(DATA_DIR / "val", processor)
    print(f">>> Train: {len(train_ds)} anh | Val: {len(val_ds)} anh")

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = SegformerForSemanticSegmentation.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        ignore_mismatched_sizes=True,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    best_val_loss = float("inf")
    OUTPUT_DIR.mkdir(exist_ok=True)
    log_path = OUTPUT_DIR / "training_log.csv"
    # Xoa log cu neu co (tranh lap dong header/du lieu tu lan chay do dang truoc)
    log_path.write_text("epoch,train_loss,val_loss\n", encoding="utf-8")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        train_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [train]")):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS

            loss.backward()
            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                optimizer.zero_grad()
            train_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS} [val]"):
                pixel_values = batch["pixel_values"].to(device)
                labels = batch["labels"].to(device)
                outputs = model(pixel_values=pixel_values, labels=labels)
                val_loss += outputs.loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        print(f"[Epoch {epoch}] train_loss={avg_train:.4f}  val_loss={avg_val:.4f}")

        # Luu log ra file de dua vao bao cao (bieu do Log Loss giong Figure 8 trong bai bao tham khao)
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(f"{epoch},{avg_train:.6f},{avg_val:.6f}\n")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            model.save_pretrained(OUTPUT_DIR / "best_model")
            processor.save_pretrained(OUTPUT_DIR / "best_model")
            print(f"  -> Luu checkpoint tot nhat (val_loss={avg_val:.4f}) tai {OUTPUT_DIR / 'best_model'}")

    print(f"\nHOAN TAT huan luyen. Checkpoint tot nhat: {OUTPUT_DIR / 'best_model'}")
    print(f"Log huan luyen day du: {OUTPUT_DIR / 'training_log.csv'} (dung ve bieu do Chuong 4)")


if __name__ == "__main__":
    main()


In [ ]:
!python scripts/05_stage2_train_segformer.py

### Đánh giá định lượng Giai đoạn 2 (mIoU, Dice, Precision, Recall)

In [ ]:
%%writefile scripts/06_evaluate_stage2.py
"""
Script: 06_evaluate_stage2.py
Muc dich: Tinh cac chi so danh gia THAT cho model SegFormer da huan luyen o
script 05 -- mIoU, Dice score, Precision, Recall tren tap validation cua
ISIC. Day la SO LIEU DINH LUONG chinh dua vao Chuong 4 khoa luan (tuong tu
cach bai bao tham khao trinh bay Confusion Matrix / Precision-Recall / F1).
"""

from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor
from tqdm import tqdm

import sys
sys.path.append(str(Path(__file__).parent))
from importlib import import_module

_stage2 = import_module("05_stage2_train_segformer".replace("-", "_"))
LesionSegDataset = _stage2.LesionSegDataset

DATA_DIR = Path("processed_isic")
CHECKPOINT_DIR = Path("segformer_melasma_checkpoint/best_model")


def compute_metrics(pred_mask: np.ndarray, gt_mask: np.ndarray):
    """pred_mask, gt_mask: mang nhi phan {0,1} cung kich thuoc."""
    tp = np.logical_and(pred_mask == 1, gt_mask == 1).sum()
    fp = np.logical_and(pred_mask == 1, gt_mask == 0).sum()
    fn = np.logical_and(pred_mask == 0, gt_mask == 1).sum()
    tn = np.logical_and(pred_mask == 0, gt_mask == 0).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    dice = (2 * tp) / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
    iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0

    return {"precision": precision, "recall": recall, "dice": dice, "iou": iou}


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    assert CHECKPOINT_DIR.exists(), f"Khong tim thay checkpoint tai {CHECKPOINT_DIR}. Chay script 05 truoc."

    processor = SegformerImageProcessor.from_pretrained(CHECKPOINT_DIR)
    model = SegformerForSemanticSegmentation.from_pretrained(CHECKPOINT_DIR).to(device)
    model.eval()

    val_ds = LesionSegDataset(DATA_DIR / "val", processor)
    val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

    all_metrics = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Danh gia"):
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].numpy()[0]

            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits  # (1, num_labels, H, W)
            upsampled = torch.nn.functional.interpolate(
                logits, size=labels.shape, mode="bilinear", align_corners=False
            )
            pred_mask = upsampled.argmax(dim=1)[0].cpu().numpy()

            all_metrics.append(compute_metrics(pred_mask, labels))

    print(f"\n=== KET QUA DANH GIA GIAI DOAN 2 (tren {len(all_metrics)} anh validation ISIC) ===")
    for key in ["precision", "recall", "dice", "iou"]:
        values = [m[key] for m in all_metrics]
        print(f"  {key.capitalize():10s}: mean = {np.mean(values):.4f}  (std = {np.std(values):.4f})")

    # Luu ra file de trich dan trong bao cao
    with open("stage2_evaluation_results.txt", "w", encoding="utf-8") as f:
        f.write("Ket qua danh gia Giai doan 2 (SegFormer fine-tuned tren ISIC 2018)\n")
        f.write(f"So anh validation: {len(all_metrics)}\n\n")
        for key in ["precision", "recall", "dice", "iou"]:
            values = [m[key] for m in all_metrics]
            f.write(f"{key}: mean={np.mean(values):.4f}, std={np.std(values):.4f}\n")
    print("\nDa luu ket qua vao stage2_evaluation_results.txt")


if __name__ == "__main__":
    main()


In [ ]:
!python scripts/06_evaluate_stage2.py

### Vẽ biểu đồ Log Loss (giống Figure 8 trong bài báo tham khảo) — dùng cho Chương 4

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log = pd.read_csv("segformer_melasma_checkpoint/training_log.csv")
plt.figure(figsize=(8,5))
plt.plot(log["epoch"], log["train_loss"], label="Train")
plt.plot(log["epoch"], log["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("SegFormer Learning Curve — Log Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig("learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Đã lưu learning_curve.png -- dùng chèn trực tiếp vào Chương 4 khóa luận.")


## 6. Giai đoạn C — Giai đoạn 3 (MASI) + Demo End-to-End trên ảnh Melasma thật

In [ ]:
%%writefile scripts/07_stage3_masi.py
"""
Script: 07_stage3_masi.py
Muc dich: GIAI DOAN 3 cua pipeline Huong 1 -- tinh chi so MASI (Melasma Area
and Severity Index) tu ket qua phan vung (mask) cua Giai doan 2, cho tung
vung giai phau da duoc khoanh vung o Giai doan 1.

CONG THUC GOC (Kimbrough-Green et al., 1994) -- CHI danh gia 4 vung: Tran,
Ma phai, Ma trai, Cam (KHONG bao gom Mui -- day la quy uoc chuan trong y
van, khong phai thieu sot cua pipeline):

    MASI = 0.3*A(f)*[D(f)+H(f)] + 0.3*A(rm)*[D(rm)+H(rm)]
         + 0.3*A(lm)*[D(lm)+H(lm)] + 0.1*A(c)*[D(c)+H(c)]

    f = forehead (tran), rm = right malar (ma phai), lm = left malar (ma trai),
    c = chin (cam)

    A (Area, dien tich ton thuong trong vung) thang diem 0-6:
        0=0% | 1=<10% | 2=10-29% | 3=30-49% | 4=50-69% | 5=70-89% | 6=90-100%
    D (Darkness, do dam mau) thang diem 0-4 (0=khong co .. 4=rat dam)
    H (Homogeneity, do dong deu) thang diem 0-4

Ghi chu quan trong (trich dan trong Chuong 4.4 khoa luan): cac nghien cuu
kiem dinh sau nay (Pandya et al., 2011) chi ra thanh phan Homogeneity (H)
kem tin cay/kem lap lai giua cac nguoi danh gia, va de xuat ban RUT GON:

    mMASI = 0.3*A(f)*D(f) + 0.3*A(rm)*D(rm) + 0.3*A(lm)*D(lm) + 0.1*A(c)*D(c)

Script nay tinh CA HAI phien ban, trong do A va D duoc uoc luong TU DONG tu
anh (thay vi bac si danh gia bang mat) -- day la dong gop/thich nghi cua
khoa luan, can neu ro la 'uoc luong tu dong, chua qua kiem dinh lam sang'
trong phan han che.
"""

from pathlib import Path
import numpy as np
import cv2

# Trong so tung vung theo cong thuc goc (KHONG co Mui)
REGION_WEIGHTS = {
    "Tran": 0.3,
    "Ma_phai": 0.3,
    "Ma_trai": 0.3,
    "Cam": 0.1,
}


def area_score(lesion_pct: float) -> int:
    """Quy doi % dien tich ton thuong sang thang diem 0-6 theo chuan MASI."""
    if lesion_pct <= 0:
        return 0
    elif lesion_pct < 10:
        return 1
    elif lesion_pct < 30:
        return 2
    elif lesion_pct < 50:
        return 3
    elif lesion_pct < 70:
        return 4
    elif lesion_pct < 90:
        return 5
    else:
        return 6


def darkness_score(lesion_pixels: np.ndarray, normal_pixels: np.ndarray) -> float:
    """Uoc luong do dam mau (0-4) bang cach so sanh do sang kenh L* (CIELAB)
    giua vung ton thuong va vung da binh thuong lan can trong cung ROI.
    Chenh lech L* cang lon (vung ton thuong toi hon) -> diem cang cao.
    """
    if len(lesion_pixels) == 0 or len(normal_pixels) == 0:
        return 0.0

    l_lesion = float(np.mean(lesion_pixels))
    l_normal = float(np.mean(normal_pixels))
    diff = max(0.0, l_normal - l_lesion)  # vung nam toi hon da thuong

    # Quy doi tuyen tinh: chenh lech L* tu 0->40 tuong ung diem 0->4
    # (Nguong 40 la uoc luong ban dau, NEN hieu chinh lai bang doi chieu
    # voi danh gia cua bac si da lieu tren mot tap mau nho neu co dieu kien.)
    score = min(4.0, (diff / 40.0) * 4.0)
    return float(round(score, 2))


def analyze_region(image_bgr: np.ndarray, mask: np.ndarray) -> dict:
    """image_bgr, mask: cung kich thuoc, mask la nhi phan {0,1} (1=vung ton thuong).
    Tra ve dict gom % dien tich, diem A, diem D."""
    lab = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2LAB)
    l_channel = lab[:, :, 0].astype(np.float32) * (100.0 / 255.0)  # quy doi ve thang L* 0-100

    total_pixels = mask.size
    lesion_pixels_mask = mask == 1
    normal_pixels_mask = mask == 0

    lesion_pct = 100.0 * float(lesion_pixels_mask.sum()) / total_pixels if total_pixels > 0 else 0.0

    a_score = area_score(lesion_pct)
    d_score = darkness_score(l_channel[lesion_pixels_mask], l_channel[normal_pixels_mask])

    return {"area_pct": float(round(lesion_pct, 2)), "A": int(a_score), "D": float(d_score)}


def compute_masi(region_results: dict) -> dict:
    """region_results: {"Tran": {"A":.., "D":..}, "Ma_phai": {...}, ...}
    (Homogeneity H duoc mac dinh = 0 do khong the tu dong uoc luong tin cay
    -- day la ly do chinh ban MASI day du can nguoi danh gia thu cong bo sung,
    trong khi mMASI khong phu thuoc H nen phu hop hon cho tu dong hoa)."""
    masi_total = 0.0
    mmasi_total = 0.0
    details = {}

    for region, weight in REGION_WEIGHTS.items():
        if region not in region_results:
            continue
        A = region_results[region]["A"]
        D = region_results[region]["D"]
        H = region_results[region].get("H", 0)  # mac dinh 0 (xem ghi chu tren)

        masi_contrib = weight * A * (D + H)
        mmasi_contrib = weight * A * D

        masi_total += masi_contrib
        mmasi_total += mmasi_contrib
        details[region] = {"A": A, "D": D, "H": H,
                            "masi_contrib": round(masi_contrib, 3),
                            "mmasi_contrib": round(mmasi_contrib, 3)}

    return {
        "MASI": round(masi_total, 3),        # thang diem ly thuyet 0-48
        "mMASI": round(mmasi_total, 3),       # thang diem ly thuyet 0-24
        "per_region": details,
    }


if __name__ == "__main__":
    # Vi du minh hoa voi du lieu gia lap -- trong pipeline that, region_results
    # se duoc dien tu ket qua thuc te cua analyze_region() cho tung ROI da
    # duoc cat o Giai doan 1 va phan vung o Giai doan 2.
    demo_regions = {
        "Tran": {"A": 3, "D": 2.1},
        "Ma_phai": {"A": 4, "D": 2.8},
        "Ma_trai": {"A": 4, "D": 2.6},
        "Cam": {"A": 1, "D": 1.0},
    }
    result = compute_masi(demo_regions)
    print("Vi du ket qua tinh MASI/mMASI (du lieu minh hoa):")
    print(result)


In [ ]:
%%writefile scripts/08_pipeline_end_to_end.py
"""
Script: 08_pipeline_end_to_end.py
Muc dich: NOI TOAN BO 3 GIAI DOAN cua Huong 1 thanh mot pipeline hoan chinh:

    Anh khuon mat day du
        -> [Giai doan 1] YOLO/Haar dinh vi mat + suy luan 5 vung ROI (giu
           nguyen do phan giai goc)
        -> [Giai doan 2] SegFormer da fine-tune phan vung ton thuong tren
           tung ROI
        -> [Giai doan 3] Tinh MASI/mMASI tu ket qua phan vung
        -> Xuat: anh minh hoa (ROI + mask chong len anh goc) + file JSON
           ket qua diem so -- day la dau ra dung de chup man hinh/dua vao
           Chuong 4 khoa luan.

CHAY:
    python 08_pipeline_end_to_end.py duong_dan_anh.jpg
"""

import json
import sys
from pathlib import Path
import cv2
import numpy as np
import torch
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor


class NpEncoder(json.JSONEncoder):
    """Lop bao ve du phong: tu dong chuyen doi moi kieu du lieu NumPy con sot
    lai (float32, int64, ndarray...) sang kieu Python thuan truoc khi ghi
    JSON, dam bao khong bao gio gap lai loi 'not JSON serializable'."""
    def default(self, obj):
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)


sys.path.append(str(Path(__file__).parent))
from importlib import import_module

_stage1 = import_module("03_stage1_roi_detection")
_stage3 = import_module("07_stage3_masi")

FaceDetector = _stage1.FaceDetector
derive_anatomical_rois = _stage1.derive_anatomical_rois
crop_roi = _stage1.crop_roi
analyze_region = _stage3.analyze_region
compute_masi = _stage3.compute_masi

CHECKPOINT_DIR = Path("segformer_melasma_checkpoint/best_model")
OUTPUT_DIR = Path("pipeline_demo_output")


def segment_roi(image_bgr, model, processor, device):
    """Chay SegFormer da fine-tune tren mot anh ROI, tra ve mask nhi phan
    {0,1} CUNG KICH THUOC voi anh ROI dau vao (upsample lai sau khi model
    du doan o do phan giai co dinh cua no)."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    inputs = processor(images=image_rgb, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    upsampled = torch.nn.functional.interpolate(
        logits, size=image_bgr.shape[:2], mode="bilinear", align_corners=False
    )
    mask = upsampled.argmax(dim=1)[0].cpu().numpy().astype(np.uint8)
    return mask


def run_pipeline(image_path: str):
    assert CHECKPOINT_DIR.exists(), (
        f"Chua co checkpoint tai {CHECKPOINT_DIR}. Chay script 05 (huan luyen) truoc."
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    processor = SegformerImageProcessor.from_pretrained(CHECKPOINT_DIR)
    model = SegformerForSemanticSegmentation.from_pretrained(CHECKPOINT_DIR).to(device)
    model.eval()

    detector = FaceDetector()

    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Khong doc duoc anh: {image_path}")

    print(">>> Giai doan 1: dinh vi khuon mat + suy luan vung ROI...")
    face_box = detector.detect(image)
    if face_box is None:
        print("[!] Khong phat hien duoc khuon mat. Dung lai.")
        return
    rois = derive_anatomical_rois(face_box)

    out_path = OUTPUT_DIR / Path(image_path).stem
    out_path.mkdir(parents=True, exist_ok=True)

    region_results = {}
    overlay = image.copy()

    print(">>> Giai doan 2: phan vung ton thuong tren tung ROI...")
    for roi in rois:
        crop = crop_roi(image, roi)
        if crop.size == 0:
            continue

        mask = segment_roi(crop, model, processor, device)
        region_results[roi.name] = analyze_region(crop, mask)

        x1, y1, x2, y2 = roi.box
        colored_mask = np.zeros_like(crop)
        colored_mask[mask == 1] = (0, 0, 255)
        blended = cv2.addWeighted(crop, 0.7, colored_mask, 0.3, 0)
        overlay[y1:y2, x1:x2] = blended
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 1)

        cv2.imwrite(str(out_path / f"{roi.name}_mask_overlay.jpg"), blended)

    print(">>> Giai doan 3: tinh chi so MASI/mMASI...")
    masi_result = compute_masi(region_results)

    cv2.imwrite(str(out_path / "full_overlay.jpg"), overlay)
    with open(out_path / "result.json", "w", encoding="utf-8") as f:
        json.dump(
            {"per_region_analysis": region_results, "masi_result": masi_result},
            f, ensure_ascii=False, indent=2, cls=NpEncoder,
        )

    print(f"\n=== KET QUA CUOI CUNG ({image_path}) ===")
    for region, data in region_results.items():
        print(f"  {region:10s}: dien tich={data['area_pct']}%  A={data['A']}  D={data['D']}")
    print(f"\n  MASI  = {masi_result['MASI']}  (thang ly thuyet 0-48)")
    print(f"  mMASI = {masi_result['mMASI']}  (thang ly thuyet 0-24, khong dung Homogeneity)")
    print(f"\nAnh + JSON ket qua da luu tai: {out_path}")
    print("-> Dung anh 'full_overlay.jpg' va noi dung 'result.json' de chup man hinh")
    print("   dua vao Chuong 4 khoa luan (minh hoa dau ra thuc te cua pipeline).")


if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Cach dung: python 08_pipeline_end_to_end.py <duong_dan_anh.jpg>")
        sys.exit(1)
    run_pipeline(sys.argv[1])


**Chạy demo trên ảnh nám thật** (từ MultiCaRe đã tải ở Bước 3.1, hoặc ảnh bạn tự upload):

In [ ]:
# Đổi đường dẫn thành ảnh nám thật bạn muốn demo:
# !python scripts/08_pipeline_end_to_end.py duong_dan_anh_nam_that.jpg
print("Sửa đường dẫn ảnh ở dòng trên rồi bỏ comment để chạy demo cuối cùng.")


## Hoàn tất

Các file quan trọng cần lưu lại để đưa vào báo cáo khóa luận:
- `segformer_melasma_checkpoint/training_log.csv` + `learning_curve.png` — Chương 4 (huấn luyện)
- `stage2_evaluation_results.txt` — Chương 4 (số liệu đánh giá mIoU/Dice/Precision/Recall)
- `pipeline_demo_output/*/full_overlay.jpg` + `result.json` — Chương 4 (minh hoạ demo thực tế + điểm MASI)
- (nếu đã làm) kết quả IoU của Giai đoạn 1 từ `04_evaluate_stage1.py evaluate`

Gửi các file/số liệu này lại để tiếp tục viết Chương 4 dựa trên kết quả thật.